# 가상 참여자 노트북 (정석: 행·열 합 특성 16개 추가 + 랜덤 포레스트)

In [1]:
import pandas as pd
import numpy as np
import joblib
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
print(train.shape, test.shape)

(1437, 66) (360, 65)


In [2]:
def display_scores(scores):
    print('Scores:', scores)
    print('Mean:', scores.mean())
    print('Standard deviation:', scores.std())

In [3]:
X, y = train.drop(columns=['id', 'label']), train['label']
X_test = test.drop(columns=['id'])

In [4]:
# 8x8 그림의 행 합 8개 + 열 합 8개 = 픽셀 64개에 0/1 가중치를 곱해 더한 값
W = np.zeros((64, 16))
for r in range(8):
    for c in range(8):
        W[r * 8 + c, r] = 1       # r번째 행의 합
        W[r * 8 + c, 8 + c] = 1   # c번째 열의 합
from sklearn.pipeline import FeatureUnion, make_pipeline
from sklearn.preprocessing import FunctionTransformer
features = FeatureUnion([('pixels', 'passthrough'), ('sums', FunctionTransformer(np.dot, kw_args={'b': W}))])
print('특성 수:', features.fit(X).transform(X.head()).shape[1])

특성 수: 80


In [5]:
from sklearn.ensemble import RandomForestClassifier
model = make_pipeline(features, RandomForestClassifier(n_estimators=300, random_state=0))

In [6]:
from sklearn.model_selection import cross_val_score
scores = cross_val_score(model, X, y, cv=10, scoring='accuracy')
display_scores(scores)

Scores: [0.96527778 1.         0.97916667 0.97222222 0.97222222 0.99305556
 0.97222222 0.97902098 0.98601399 0.96503497]
Mean: 0.9784236596736597
Standard deviation: 0.010979970751621581


In [7]:
model.fit(X, y)
pred = model.predict(X_test)

In [8]:
pd.DataFrame({'id': test['id'], 'label': pred}).to_csv('submission.csv', index=False)
joblib.dump(model, 'model.pkl')

['model.pkl']